# Week 3 Day 4 - Lab 2: UnderCover on Runtime

## Deploying the agent beside the app

Today's build made the agent an AgentCore app, which runs the same way on your laptop and on AgentCore Runtime. The app now starts each dispatch and evaluation in a Runtime session of its own and keeps one at a time itself, and a senior handler can send a large approval back to the agent with a note. In this lab we bring back Day 3's pieces on AWS, deploy the agent on Runtime, point the app at it, and run a swarm of 25 claim agents in one session from the deployed dashboard.

## 6 parts:

PART 1: Day 3's pieces again

PART 2: The agent's project

PART 3: Deploying the agent

PART 4: Pointing the app at the agent

PART 5: A swarm on Runtime, and a claim sent back

PART 6: Teardown

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">You need Lab 1's setup, with the AgentCore CLI at 1.0.0-rc.5, and today's build of the product, yours or <code>week3day4-end</code>, with <code>OPENROUTER_API_KEY</code>, <code>DATABASE_URL</code>, <code>VITE_SUPABASE_URL</code> and <code>UNDERCOVER_API_KEY</code> in your <code>.env</code> file. Keep a terminal open in the <code>undercover</code> folder for the CLI's commands.<br/><br/>The app and the context layer are inside AWS's free allowance for Lambda. The Gateway, the Policy, the Memory and the four secrets cost a fraction of a cent while they exist. Runtime charges only while a session is working, so the swarm costs about a cent on AWS and a few cents on OpenRouter. Part 5 also reseeds your database, which starts the queue, the scoreboard and the configuration's versions afresh. Part 6 deletes everything except the two functions.<br/><br/>If you're on the free route, <a href="week3day4_lab2free.ipynb">week3day4_lab2free.ipynb</a> is a guide to running the agent on Render beside the app.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Watching your AWS spend</h2>
            <span style="color:#118AB2;">Before you start, open a terminal in the <code>undercover</code> folder and run <code>uv run monitor.py</code>. It opens a page in your browser with what your AWS account has spent today and over the last 30 days, with and without your credits. AWS updates the figures up to three times a day.<br/><br/>For the official source of truth, please review Billing and Cost Management in your AWS console. Always have budgets with billing alerts, like the one from Day 1.
            </span>
        </td>
    </tr>
</table>

In [ ]:
import json
import os
import time
import uuid
from pathlib import Path

import boto3
import psycopg
from dotenv import load_dotenv, set_key, unset_key

import week3day4_aws as aws

## Connect to AWS

boto3 uses the access key that `aws configure` saved on Day 1, in us-east-1.

In [ ]:
load_dotenv(override=True)

session = boto3.Session()
region = session.region_name
identity = session.client("sts").get_caller_identity()
account = identity["Account"]

print(f"Working as {identity['Arn']} in {region}")

In [ ]:
lambda_client = session.client("lambda")
iam = session.client("iam")
logs = session.client("logs")
agentcore_runtime = session.client("bedrock-agentcore")

In [ ]:
for name in ["OPENROUTER_API_KEY", "DATABASE_URL", "VITE_SUPABASE_URL", "UNDERCOVER_API_KEY"]:
    print(f"{name}: {'found' if os.getenv(name) else 'missing, please add it to .env'}")

API_KEY = os.environ["UNDERCOVER_API_KEY"]
# Supabase's transaction pooler, which the app on Lambda and the agent on Runtime share
POOLED_DATABASE_URL = os.environ["DATABASE_URL"].replace(":5432/", ":6543/")

## Part 1: Day 3's pieces again

Day 3's lab deployed the app and the context layer on Lambda, and created the Gateway with both as targets, the Policy and the Memory. Its teardown kept the two functions and deleted the rest. `week3day4_aws.py`, beside this notebook, holds Day 3's code as functions, so each piece takes one call today. Each creates what's missing and updates what's there.

### The app with today's routes

Today's build added the send back and the app's own record of what's running, so the app needs the new code. We build the dashboard, then update the function in place, which keeps its address.

In [ ]:
!cd .. && npm run build

In [ ]:
app_environment = {
    "DATABASE_URL": POOLED_DATABASE_URL,
    "VITE_SUPABASE_URL": os.environ["VITE_SUPABASE_URL"],
    "UNDERCOVER_API_KEY": API_KEY,
}

app_url = aws.deploy_app(app_environment)
print(app_url)

### The context layer

The context layer hasn't changed since Day 3, and deploying it again makes sure it's there.

In [ ]:
context_url = aws.deploy_context()
print(context_url)

### The Gateway and the Policy

The Gateway with the app and the context layer as its targets, the app's API key in AgentCore Identity, and the Policy enforced, with its forbid on any payment over $1,000. It takes about a minute.

In [ ]:
gateway = aws.create_gateway(app_url, context_url, API_KEY)
gateway_url = gateway["gatewayUrl"]
print(gateway_url)

### The Memory

The agent's memory of each customer, with its Semantic and Summary strategies. It takes about two and a half minutes.

In [ ]:
memory_id = aws.create_memory()

## Part 2: The agent's project

### What's in the repo

The repo has an AgentCore project like Lab 1's, with its configuration in `agentcore/`. Please open `agentcore/agentcore.json`, `agent/entrypoint.py` and `runtime/main.py` in your editor.

`agentcore/agentcore.json` declares one runtime, `agent`, built as a zip from `runtime/`, and three credentials, each named after the environment variable the agent reads it from.

`agent/entrypoint.py` is the agent's AgentCore app, made with `BedrockAgentCoreApp` as in Lab 1's hello world. Its one entrypoint takes a payload that names an operation, such as `{"operation": "work_run", "run_id": ...}` from the app. Each operation answers at once and does its work on a thread of its own, which it registers with the AgentCore app as an async task. While a task is registered, the ping tells Runtime the session is busy, so a dispatch carries on after the app has its answer, and the session ends once the work is done. `npm run dev` runs the same app on your laptop, on port 8080.

On its first request, the agent fetches the three credentials from AgentCore Identity and opens its pool. As in Lab 1, nothing happens at import, since V2 starts every session from a snapshot.

`runtime/main.py` is the file Runtime starts, and it runs the agent's app. The deploy packs `runtime/` with `agent/` beside it, and installs the `runtime` dependency group from `pyproject.toml` at the versions in `uv.lock`. `agentcore/cdk/lib/cdk-stack.ts` does this at each deploy. The runtime's own role may call the Gateway and use the Memory, as `runtime/policy.json` sets out.

### The CDK project's packages

As in Lab 1, the CLI deploys with the CDK project in `agentcore/cdk`, which needs its packages installed once. In your terminal, in the `undercover` folder, run:

`npm ci --prefix agentcore/cdk`

### The secrets and the settings

The CLI reads each credential's value from `agentcore/.env.local`, as in Lab 1. Git ignores the file. We write the three values there, with the database on the transaction pooler, since each Runtime session has connections of its own.

The deploy also reads three settings from the same file and gives them to the runtime as environment variables: the app's address, the Gateway's and the Memory's. On Runtime the agent's tools go through the Gateway, since only the Gateway can reach the context layer.

In [ ]:
LOCAL_ENV = Path("../agentcore/.env.local")
LOCAL_ENV.touch()

secrets = {
    "AGENTCORE_CREDENTIAL_OPENROUTER_API_KEY": os.environ["OPENROUTER_API_KEY"],
    "AGENTCORE_CREDENTIAL_UNDERCOVER_API_KEY": API_KEY,
    "AGENTCORE_CREDENTIAL_DATABASE_URL": POOLED_DATABASE_URL,
}
settings = {
    "UNDERCOVER_API": app_url,
    "UNDERCOVER_GATEWAY_URL": gateway_url,
    "UNDERCOVER_MEMORY_ID": memory_id,
}

for name, value in (secrets | settings).items():
    set_key(LOCAL_ENV, name, value)

print(f"Wrote {', '.join(secrets | settings)} to {LOCAL_ENV}")

## Part 3: Deploying the agent

In your terminal, in the `undercover` folder, run:

`agentcore deploy`

It takes about 7 minutes. The CDK bootstrap from Lab 1 is still there, so this deploy makes the three credentials in AgentCore Identity, and the stack `AgentCore-undercover-default` with the runtime and its role. As in Lab 1, the project's CDK code puts the runtime on platform version V2, so most of the time goes on the snapshot.

In [ ]:
runtime = aws.find_runtimes("undercover_default_agent")[0]
runtime_arn = runtime["agentRuntimeArn"]
print(runtime_arn)

### Trying it

The app sends each request as a payload that names an operation, in a new session each time. A health check, `{"operation": "health"}`, shows the agent getting ready in a new session: it fetches the secrets from Identity and opens its pool before it answers.

In [ ]:
def call_agent(payload):
    """Send a payload to the agent on Runtime in a new session, as the app does."""
    started = time.time()
    response = agentcore_runtime.invoke_agent_runtime(
        agentRuntimeArn=runtime_arn, runtimeSessionId=f"lab-{uuid.uuid4()}", payload=json.dumps(payload)
    )
    reply = json.loads(response["response"].read())
    print(f"{time.time() - started:.1f} seconds: {reply}")


call_agent({"operation": "health"})

## Part 4: Pointing the app at the agent

Two changes to the app on Lambda. Its role may now invoke the runtime, and nothing else on Runtime. And `UNDERCOVER_AGENT_RUNTIME` tells it to start the agent's work there, rather than on a laptop.

In [ ]:
invoke_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": "bedrock-agentcore:InvokeAgentRuntime",
            "Resource": [runtime_arn, f"{runtime_arn}/runtime-endpoint/*"],
        }
    ],
}

iam.put_role_policy(RoleName=aws.APP_ROLE_NAME, PolicyName="undercover-app-runtime", PolicyDocument=json.dumps(invoke_policy))
print(f"{aws.APP_ROLE_NAME} may invoke the agent's runtime")

In [ ]:
app_environment["UNDERCOVER_AGENT_RUNTIME"] = runtime_arn

lambda_client.update_function_configuration(FunctionName=aws.APP_FUNCTION, Environment={"Variables": app_environment})
lambda_client.get_waiter("function_updated_v2").wait(FunctionName=aws.APP_FUNCTION)
print(f"{aws.APP_FUNCTION} starts the agent's work on {runtime_arn}")

## Part 5: A swarm on Runtime, and a claim sent back

### A fresh queue

We reseed, so all 26 claims are waiting.

In [ ]:
!cd .. && npm run seed

### The swarm

Open the deployed dashboard and sign in as Dana Whitfield, who can sign off over the limit. On the queue, start a dispatch with 25 claim agents. The app creates the run, starts it in one Runtime session and has its answer at once, while the session works on: the dispatcher and the 25 claim agents, each claim's tools and payment through the Gateway, and each claim's progress saved in Supabase.

In [ ]:
print(app_url)

Once you've started it, this cell counts the queue's claims by status every 15 seconds, straight from Supabase, until none is waiting or in progress, for up to 20 minutes.

In [ ]:
COUNT_CLAIMS = "SELECT status, count(*) FROM undercover.claims WHERE evaluation_id IS NULL GROUP BY status ORDER BY status"

with psycopg.connect(os.environ["DATABASE_URL"], autocommit=True) as connection:
    for check in range(80):
        counts = dict(connection.execute(COUNT_CLAIMS).fetchall())
        print(", ".join(f"{status} {number}" for status, number in counts.items()))
        if not counts.keys() & {"waiting", "in_progress"}:
            break
        time.sleep(15)

Runtime keeps each session's output in CloudWatch. In your terminal, `agentcore log runtime --since 15m` shows the runtime's logs.

### Sending a claim back

The three approvals over the limit wait for sign-off. Open one of them, look at the agent's decision, and send it back with a note saying what to look at again. On CLM-1011, for example, Meridian Air has already paid the customer $350 for the lost bag, and a note could ask the agent to check that it was taken off.

The app keeps the decision that was sent back, marks the claim as sent back and starts the agent on it in a new Runtime session. The session takes the claim, loads its thread from Supabase, and the agent reads the note and decides again. Refresh the claim's page to watch its steps.

Then sign off one of the others. The app pays it, and a short session finishes the agent's work on the claim, saving it to the Memory.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">One call for each piece of work</h2>
            <span style="color:#118AB2;">The app made one call to Runtime for the whole dispatch, and the session ran the dispatcher and 25 claim agents together, as the agent does on your laptop. The send back and the sign-off were each a call of their own, in a new session, which picked up the claim where the first session left it. Everything the agent needs between sessions is in Supabase.
            </span>
        </td>
    </tr>
</table>

## Part 6: Teardown

### The app goes back to the laptop route

We take the setting and the permission off the app. Until the agent is deployed again, the deployed dashboard can't reach the agent.

In [ ]:
del app_environment["UNDERCOVER_AGENT_RUNTIME"]

lambda_client.update_function_configuration(FunctionName=aws.APP_FUNCTION, Environment={"Variables": app_environment})
lambda_client.get_waiter("function_updated_v2").wait(FunctionName=aws.APP_FUNCTION)
iam.delete_role_policy(RoleName=aws.APP_ROLE_NAME, PolicyName="undercover-app-runtime")
print(f"{aws.APP_FUNCTION} no longer calls the runtime")

### The agent

As in Lab 1, `agentcore remove all` empties the project's configuration and the next deploy deletes everything from AWS. The repo's configuration is what Day 5 deploys again, so we put it back from git afterwards. In your terminal, in the `undercover` folder, run:

`agentcore remove all --yes`

`agentcore deploy --yes`

`git restore agentcore/agentcore.json`

Then take the three settings out of `agentcore/.env.local`. `remove all` has already taken out the secrets.

In [ ]:
for name in settings:
    unset_key(LOCAL_ENV, name)

print(f"Took {', '.join(settings)} out of {LOCAL_ENV}")

The runtime's log group keeps its logs forever, so we delete it by name.

In [ ]:
LOG_GROUPS = "/aws/bedrock-agentcore/runtimes/undercover_default_agent"

for page in logs.get_paginator("describe_log_groups").paginate(logGroupNamePrefix=LOG_GROUPS):
    for group in page["logGroups"]:
        logs.delete_log_group(logGroupName=group["logGroupName"])
        print(f"Deleted log group {group['logGroupName']}")

### Day 3's pieces

The Gateway with its targets, the Policy, the app's key in AgentCore Identity, the Memory and the Gateway's role, as at the end of Day 3. The two functions stay.

In [ ]:
aws.tear_down()

### The CDK bootstrap

The bootstrap's KMS key costs $1 a month. We can delete it here and it will be recreated with the next deploy. Deleting it takes about a minute.

In [ ]:
aws.delete_bootstrap()

If you're stopping the course here, set `DELETE_THE_FUNCTIONS` to `True` and run the cell to delete the app and the context layer from Lambda as well, with their roles and log groups.

In [ ]:
DELETE_THE_FUNCTIONS = False

if DELETE_THE_FUNCTIONS:
    aws.delete_functions()

In [ ]:
time.sleep(10)
aws.show_resources()

Over the next day or two, keep an eye on your spend with `uv run monitor.py`. For the official figures, review Billing and Cost Management in your AWS console.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">Challenge: run an evaluation of the held-out set from the deployed dashboard with the agent on Runtime, and compare its time per claim and its cost with the same evaluation on your laptop. Then look at the session in the runtime's logs.<br/><br/>I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>